# Yellowhammer Hybrid PINN vs. Random Forest --- Count Prediction

Reproducible notebook for the count-prediction experiment (adaptive-gated
hybrid PINN benchmarked against a Random Forest) of the paper
*A Distributed Parameter Identification Approach for Habitat-Modulated
Population Dynamics via Physics-Informed Neural Networks*.

This is the tuned configuration used for the reported results
(gate lower bound $0.15$, fixed $\lambda_{\mathrm{PDE}}=0.05$, anchor weight $0.03$).

**Data (place next to the notebook or upload in Colab):**
`Yellowhammer_Clim_Bioclim_CLC_2002_2024.csv`, `Data_Estimation_70_V1.csv`,
`Data_Validation_30_V1.csv` --- from the Yellowhammer benchmark
(https://github.com/lias-laboratory/yellowhammer-benchmark).

**Mode:** set the run mode to `pilot` (10 draws, fast) or `final` (100 draws).
A GPU runtime is recommended.


In [ ]:
# Colab already provides these packages. The command is harmless when they exist.
%pip install -q numpy pandas scipy scikit-learn matplotlib



In [ ]:
from __future__ import annotations

import copy
import io
import json
import math
import os
import random
import shutil
import time
import urllib.request
import zipfile
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib.path import Path as MplPath
from scipy.interpolate import RegularGridInterpolator
from scipy.io import loadmat
from scipy.ndimage import distance_transform_edt, gaussian_filter
from scipy.stats import qmc
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import PoissonRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")
print("torch", torch.__version__, "| device", DEVICE)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def FIT(y_true, y_pred) -> float:
    """Official LIAS fitting criterion, in percent."""
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    den = np.linalg.norm(y_true - y_true.mean())
    return np.nan if den == 0 else 100.0 * (1.0 - np.linalg.norm(y_true - y_pred) / den)




## 1. Download and validate the exact benchmark data

The small `Yellowhammer_2002_2024.csv` file has only five columns and cannot
reproduce the report's GLM/GAM/RF comparison. The official complete file has
115 columns, including climate, BIO and CLC predictors.



In [ ]:
DATA_DIR = Path("/content/yellowhammer_official" if Path("/content").exists() else "yellowhammer_official")
DATA_DIR.mkdir(parents=True, exist_ok=True)

OFFICIAL_URLS = {
    "Yellowhammer_Clim_Bioclim_CLC_2002_2024.csv": (
        "https://forge.lias-lab.fr/datasets/yellowhammer-benchmark/"
        "Yellowhammer_Clim_Bioclim_CLC_2002_2024.csv.zip"
    ),
    "Data_Estimation_70_V1.csv": (
        "https://forge.lias-lab.fr/datasets/yellowhammer-benchmark/"
        "Data_Estimation_70_V1.csv.zip"
    ),
    "Data_Validation_30_V1.csv": (
        "https://forge.lias-lab.fr/datasets/yellowhammer-benchmark/"
        "Data_Validation_30_V1.csv.zip"
    ),
}


def download_zip_member(url: str, expected_name: str, out_dir: Path) -> Path:
    target = out_dir / expected_name
    if target.exists() and target.stat().st_size > 0:
        return target

    # Prefer an uploaded/local official file. In Colab this finds a file placed
    # directly in /content; locally it also searches the current project tree.
    roots = []
    for root in [Path.cwd(), Path("/content")]:
        if root.exists() and root.resolve() not in [r.resolve() for r in roots]:
            roots.append(root)
    for root in roots:
        for candidate in root.rglob(expected_name):
            if candidate.resolve() == target.resolve() or not candidate.is_file():
                continue
            shutil.copy2(candidate, target)
            print("using uploaded/local", candidate)
            return target

    print("downloading", expected_name)
    with urllib.request.urlopen(url, timeout=120) as response:
        payload = response.read()
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        candidates = [n for n in archive.namelist() if Path(n).name == expected_name]
        if not candidates:
            raise FileNotFoundError(f"{expected_name} is absent from {url}")
        with archive.open(candidates[0]) as src, target.open("wb") as dst:
            shutil.copyfileobj(src, dst)
    return target


for filename, url in OFFICIAL_URLS.items():
    download_zip_member(url, filename, DATA_DIR)

FULL_CSV = DATA_DIR / "Yellowhammer_Clim_Bioclim_CLC_2002_2024.csv"
EST_CSV = DATA_DIR / "Data_Estimation_70_V1.csv"
VAL_CSV = DATA_DIR / "Data_Validation_30_V1.csv"

df = pd.read_csv(FULL_CSV)
assert len(df) == 22_641, f"Expected 22,641 rows, found {len(df):,}"
assert len(df.columns) == 115, (
    f"Expected the complete 115-column file, found {len(df.columns)} columns. "
    "Do not use the five-column count-only file for this benchmark."
)


def read_official_splits(path: Path) -> list[np.ndarray]:
    """Read ragged, semicolon-delimited, 1-based indices with NO header."""
    rows = []
    for line_no, line in enumerate(path.read_text().splitlines(), start=1):
        values = [int(x) for x in line.split(";") if x.strip()]
        if not values:
            raise ValueError(f"Empty split at {path}:{line_no}")
        idx = np.asarray(values, dtype=np.int64) - 1
        if idx.min() < 0 or idx.max() >= len(df):
            raise IndexError(f"Out-of-range index at {path}:{line_no}")
        rows.append(idx)
    if len(rows) != 100:
        raise ValueError(f"Expected 100 splits in {path}, found {len(rows)}")
    return rows


EST_SPLITS = read_official_splits(EST_CSV)
VAL_SPLITS = read_official_splits(VAL_CSV)

all_rows = np.arange(len(df))
for k, (tr, va) in enumerate(zip(EST_SPLITS, VAL_SPLITS)):
    if np.intersect1d(tr, va).size:
        raise ValueError(f"Draw {k}: train/validation overlap")
    if not np.array_equal(np.sort(np.r_[tr, va]), all_rows):
        raise ValueError(f"Draw {k}: train/validation do not partition all rows")

print("complete data:", df.shape)
print("official draws:", len(EST_SPLITS))
print("draw 0:", len(EST_SPLITS[0]), "train |", len(VAL_SPLITS[0]), "validation")
print("zeros: %.1f%%" % (100 * (df.EMBCIT == 0).mean()))




## 2. Load the habitat used in the PINN report

Upload `u_GLM_Bruant_jaune.mat` and `Map_France.mat` when Colab asks. The class
below reproduces the supplied habitat notebook:

\[
H_0=u_{GLM}/15,\quad Z=\operatorname{logit}(H_0),\quad
H=\operatorname{sigmoid}(\widetilde Z).
\]



In [ ]:
# Locate data files (works in Colab, Jupyter, or plain local run)
import os, sys
def in_colab():
    try:
        import google.colab  # noqa
        return True
    except ImportError:
        return False
NEEDED = ["Yellowhammer_Clim_Bioclim_CLC_2002_2024.csv",
          "Data_Estimation_70_V1.csv", "Data_Validation_30_V1.csv"]
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    if in_colab():
        from google.colab import files
        print("Please upload:", missing); files.upload()
    else:
        sys.exit("Missing data files: %s\nDownload from the Yellowhammer benchmark:\n"
                 "  https://github.com/lias-laboratory/yellowhammer-benchmark" % missing)
print("All data files present.")


## 3. Official predictors and training-only statistical offset

The report's RF uses latitude, longitude, BIO1, BIO2 and nine selected CLC
variables. We add `Year` because time is an explicit PINN coordinate. The
Poisson GLM offset is fitted separately inside every 70% training set.



In [ ]:
CLC9 = ["CLC231", "CLC211", "CLC242", "CLC311", "CLC112",
        "CLC243", "CLC312", "CLC313", "CLC324"]
RF_FEATURES = ["Latitude", "Longitude", "BIO1", "BIO2"] + CLC9
HYBRID_FEATURES = RF_FEATURES + ["Year"]
GLM_LINEAR = ["Longitude", "Latitude", "BIO1", "BIO2"] + CLC9 + ["Year"]
GLM_SQUARED = ["BIO1", "BIO2"] + CLC9

missing = sorted(set(HYBRID_FEATURES + ["EMBCIT"]) - set(df.columns))
assert not missing, f"Missing official variables: {missing}"
assert not df[HYBRID_FEATURES + ["EMBCIT"]].isna().any().any(), "NaNs in selected data"


class TrainingOnlyGLM:
    """GLM4-like Poisson offset, fitted with training rows only."""

    def __init__(self, alpha=1e-3):
        self.scaler = StandardScaler()
        self.model = PoissonRegressor(alpha=alpha, max_iter=1500, tol=1e-8)

    def _design(self, frame, fit=False):
        raw = frame[GLM_LINEAR].to_numpy(float)
        z = self.scaler.fit_transform(raw) if fit else self.scaler.transform(raw)
        positions = [GLM_LINEAR.index(c) for c in GLM_SQUARED]
        return np.column_stack([z, z[:, positions] ** 2])

    def fit(self, frame, y):
        self.model.fit(self._design(frame, fit=True), np.asarray(y, float))
        return self

    def predict(self, frame):
        return np.clip(self.model.predict(self._design(frame)), 1e-4, 40.0)


def fit_random_forest(frame, y, seed, smoke=False, use_year=True):
    """Random Forest control, fitted only on the current outer training rows.

    With `use_year=True` the RF receives the same tabular predictors available
    to the observation network. Set it to False to reproduce the report's
    13-variable input list (the algorithm remains scikit-learn RF, so exact
    tree realizations can differ from R's `randomForest` package).
    """
    features = HYBRID_FEATURES if use_year else RF_FEATURES
    model = RandomForestRegressor(
        n_estimators=100 if smoke else 500,
        max_features=max(1, len(features) // 3),
        min_samples_leaf=5,
        bootstrap=True,
        n_jobs=-1,
        random_state=seed,
    )
    model.fit(frame[features], y)
    model.feature_names_used_ = features
    return model




## 4. Physics pool and hybrid architecture

The latent state follows the report's habitat-modulated reaction-diffusion
equation. A separate observation head maps the latent ecological state and
BIO/CLC covariates to the expected single-visit count. This separation is the
central correction: a smooth population density and a noisy observed count are
not the same random variable. Unlike Experiment 1, the physics contribution is
local rather than global:

\[
w_i=\sigma\!\left(g_\phi(z_i,H_i,t_i,x_i,y_i)\right),
\qquad
\log\mu_i=(1-w_i)\log\mu_i^{GLM}
+w_i\log(15n_\theta)+\delta_\phi(z_i).
\]

`w_i` is reported on every draw. If it collapses near zero, the observation
model is ignoring the physical state and no physics-based predictive claim is
justified.


The missing initial-condition term is now explicit. At 2,500 points
inside mainland France, with $t_0=2002$, the normalized latent state is
matched to the first supplied denoised field slice:

\[
\mathcal L_{IC}=\frac{1}{N_{IC}}\sum_{j=1}^{N_{IC}}
\left[n_\theta(0,x_j,y_j)-\frac{u_{GLM}(2002,x_j,y_j)}{15}\right]^2.
\]

This is `paper-IC`: it reproduces the report and is not claimed to be an
independent, leakage-free initial map.


In [ ]:
@dataclass
class RunConfig:
    mode: str = "smoke"       # smoke, pilot, final
    seed: int = 20260827
    fourier_m: int = 64
    width: int = 128
    depth: int = 4
    data_batch: int = 2048
    physics_batch: int = 768
    boundary_batch: int = 256
    initial_batch: int = 512
    pool_collocation: int = 20_000
    pool_boundary: int = 3_000
    pool_initial: int = 2_500
    lr_pretrain: float = 2e-3
    lr_ramp: float = 1.2e-3
    lr_refine: float = 8e-4
    weight_decay: float = 2e-5
    lambda_zinb: float = 0.05
    lambda_anchor: float = 0.03  # reduced: let the physical field express itself
    lambda_pde: float = 0.05
    # Report value; L_IC is dimensionless because both state and target
    # are divided by DENSITY_SCALE=15.
    lambda_ic: float = 3.0
    lambda_bc: float = 0.005
    lambda_gate_prior: float = 0.0
    gate_prior_mean: float = 0.35
    pretrain_epochs: int = 40
    physics_ramp_epochs: int = 60
    refine_epochs: int = 100
    eval_every: int = 25
    pde_candidates: tuple[float, ...] = (0.05,)
    tune_pretrain_epochs: int = 40
    tune_ramp_epochs: int = 60
    tune_refine_epochs: int = 100
    inner_fraction: float = 0.12
    use_lbfgs: bool = False
    lbfgs_steps: int = 25
    lbfgs_data: int = 4096
    lbfgs_physics: int = 1536
    lbfgs_boundary: int = 384
    lbfgs_initial: int = 1_536
    run_physics_ablation: bool = False
    rf_use_year: bool = True

    def configure_mode(self):
        if self.mode == "smoke":
            self.pretrain_epochs = 40
            self.physics_ramp_epochs = 60
            self.refine_epochs = 100
            self.eval_every = 25
            self.pde_candidates = (0.05,)
            self.pool_collocation = 8_000
            self.pool_boundary = 1_200
            self.lbfgs_steps = min(self.lbfgs_steps, 3)
        elif self.mode == "pilot":
            self.pretrain_epochs = 200
            self.physics_ramp_epochs = 300
            self.refine_epochs = 700
            self.tune_pretrain_epochs = 100
            self.tune_ramp_epochs = 150
            self.tune_refine_epochs = 350
            self.eval_every = 50
            self.pde_candidates = (0.05,)  # fixed: FIT-based calibration drove lambda to its minimum
            self.lbfgs_steps = min(self.lbfgs_steps, 10)
        elif self.mode == "final":
            # Phase 3 is deliberately long. Increase refine_epochs to 3500 only
            # if the inner-validation pilot still improves after epoch 2500.
            self.pretrain_epochs = 600
            self.physics_ramp_epochs = 600
            self.refine_epochs = 2_500
            self.tune_pretrain_epochs = 150
            self.tune_ramp_epochs = 200
            self.tune_refine_epochs = 500
            self.eval_every = 50
            self.pde_candidates = (0.05,)  # fixed: FIT-based calibration drove lambda to its minimum
        else:
            raise ValueError("mode must be smoke, pilot or final")
        return self

    @property
    def adam_epochs(self):
        return self.pretrain_epochs + self.physics_ramp_epochs + self.refine_epochs


# Change these values, restart the runtime, then Run all.
RUN_MODE = "final"                 # "smoke", "pilot", or "final"
USE_LBFGS = False                  # Enable only after it improves inner validation.
RUN_PHYSICS_ABLATION = False       # True doubles neural training cost.
RF_USE_YEAR = True                 # Fair RF: same tabular inputs as the PINN head.

CFG = RunConfig(
    mode=RUN_MODE,
    use_lbfgs=USE_LBFGS,
    run_physics_ablation=RUN_PHYSICS_ABLATION,
    rf_use_year=RF_USE_YEAR,
).configure_mode()


def lhs_interior(n, seed):
    engine = qmc.LatinHypercube(d=3, seed=seed)
    out = []
    while sum(len(x) for x in out) < n:
        s = engine.random(max(2000, 3 * (n - sum(len(x) for x in out))))
        year = 2002.0 + TIME * s[:, 0]
        x = habitat.x_min + (habitat.x_max - habitat.x_min) * s[:, 1]
        y = habitat.y_min + (habitat.y_max - habitat.y_min) * s[:, 2]
        keep = habitat.inside(x, y)
        out.append(np.column_stack([year[keep], x[keep], y[keep]]))
    return np.concatenate(out)[:n]


def boundary_candidates():
    result = []
    for xs, ys in _segments(habitat.xfr, habitat.yfr):
        tx, ty = np.gradient(xs), np.gradient(ys)
        norm = np.hypot(tx, ty) + 1e-12
        nx, ny = ty / norm, -tx / norm
        eps = 1500.0
        correct = habitat.inside(xs + eps * nx, ys + eps * ny)
        nx, ny = np.where(correct, nx, -nx), np.where(correct, ny, -ny)
        bx, by = xs + eps * nx, ys + eps * ny
        keep = habitat.inside(bx, by)
        result.append(np.column_stack([bx[keep], by[keep], nx[keep], ny[keep]]))
    return np.concatenate(result)


def make_initial_condition_pool(cfg):
    """Paper initial condition: supplied u_GLM field at t0=2002.

    Points are sampled once from valid mainland-France grid cells. The
    target is the dimensionless state u_GLM/15, exactly matching the
    StateNet output scale used in the PDE residual.
    """
    u0 = np.asarray(habitat.u_glm[:, :, 0], dtype=float)
    valid = habitat.mask & np.isfinite(u0)
    rows, cols = np.where(valid)
    if len(rows) == 0:
        raise ValueError("No valid mainland-France points for the IC pool")
    rng = np.random.default_rng(cfg.seed + 17_003)
    take = rng.choice(
        len(rows), size=cfg.pool_initial, replace=len(rows) < cfg.pool_initial
    )
    rr, cc = rows[take], cols[take]
    x = habitat.lon_axis[cc]
    y = habitat.lat_axis[rr]
    return {
        "txy": np.column_stack([
            np.zeros(len(take)),
            (x - habitat.x_min) / LENGTH,
            (y - habitat.y_min) / LENGTH,
        ]).astype(np.float32),
        "target": (u0[rr, cc] / DENSITY_SCALE).astype(np.float32)
            .reshape(-1, 1),
    }


def make_physics_pool(cfg):
    points = lhs_interior(cfg.pool_collocation, cfg.seed)
    year, x, y = points.T
    h, hx, hy = habitat.query(year, x, y)
    txy = np.column_stack([
        (year - 2002.0) / TIME,
        (x - habitat.x_min) / LENGTH,
        (y - habitat.y_min) / LENGTH,
    ]).astype(np.float32)
    # Convert physical habitat gradients (per metre) to normalized-coordinate gradients.
    col = {
        "txy": txy,
        "h": h.astype(np.float32),
        "hx": (hx * LENGTH).astype(np.float32),
        "hy": (hy * LENGTH).astype(np.float32),
    }

    candidates = boundary_candidates()
    rng = np.random.default_rng(cfg.seed)
    take = rng.choice(len(candidates), cfg.pool_boundary, replace=len(candidates) < cfg.pool_boundary)
    bx, by, nx, ny = candidates[take].T
    years = rng.uniform(2002.0, 2024.0, len(take))
    bc = {
        "txy": np.column_stack([
            (years - 2002.0) / TIME,
            (bx - habitat.x_min) / LENGTH,
            (by - habitat.y_min) / LENGTH,
        ]).astype(np.float32),
        "normal": np.column_stack([nx, ny]).astype(np.float32),
    }
    return col, bc


PHYSICS_POOL, BOUNDARY_POOL = make_physics_pool(CFG)
INITIAL_POOL = make_initial_condition_pool(CFG)
print(
    "physics pool:", len(PHYSICS_POOL["txy"]), "interior |",
    len(INITIAL_POOL["txy"]), "initial |",
    len(BOUNDARY_POOL["txy"]), "boundary",
)


class StateNet(nn.Module):
    def __init__(self, width=128, depth=4, fourier_m=64, sigma=5.0, seed=0):
        super().__init__()
        generator = torch.Generator().manual_seed(seed)
        self.register_buffer("B", torch.randn(2, fourier_m, generator=generator) * sigma)
        dims = [3 + 2 * fourier_m] + [width] * depth + [1]
        layers = []
        for a, b in zip(dims[:-2], dims[1:-1]):
            layers += [nn.Linear(a, b), nn.Tanh()]
        layers += [nn.Linear(dims[-2], dims[-1])]
        self.net = nn.Sequential(*layers)
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, txy):
        xy = txy[:, 1:3]
        projection = 2.0 * math.pi * (xy @ self.B)
        features = torch.cat([txy, torch.sin(projection), torch.cos(projection)], dim=1)
        return F.softplus(self.net(features)) + 1e-5


class HybridPINN(nn.Module):
    def __init__(self, n_env, cfg, seed):
        super().__init__()
        self.state = StateNet(cfg.width, cfg.depth, cfg.fourier_m, seed=seed)
        self.observation_trunk = nn.Sequential(
            nn.Linear(n_env, 96), nn.SiLU(), nn.Dropout(0.15),
            nn.Linear(96, 64), nn.SiLU(), nn.Dropout(0.10),
        )
        self.correction_head = nn.Linear(64, 1)
        self.zero_head = nn.Linear(64, 1)
        # g_phi receives the encoded covariates plus explicit (t,x,y,H).
        self.physics_gate = nn.Sequential(
            nn.Linear(64 + 4, 48), nn.SiLU(), nn.Dropout(0.05),
            nn.Linear(48, 1),
        )
        # Start near w=0.35, but let every observation learn its own value.
        nn.init.zeros_(self.physics_gate[-1].weight)
        nn.init.constant_(
            self.physics_gate[-1].bias,
            math.log(cfg.gate_prior_mean / (1.0 - cfg.gate_prior_mean)),
        )
        # Constrained inverse parameters; Kmax is anchored as in the paper.
        self.pD = nn.Parameter(torch.tensor(-2.0))
        self.pa = nn.Parameter(torch.tensor(0.0))
        self.pr = nn.Parameter(torch.tensor(-1.2))
        self.pb = nn.Parameter(torch.tensor(-1.5))
        self.pg = nn.Parameter(torch.tensor(-0.2))
        self.raw_theta = nn.Parameter(torch.tensor(0.5))

    def physics_parameters(self):
        return {
            "D0": F.softplus(self.pD),
            "alpha": torch.sigmoid(self.pa),
            "rmax": F.softplus(self.pr),
            "beta": torch.sigmoid(self.pb),
            "Kmax": torch.tensor(1.2, device=self.pD.device),
            "gamma": torch.sigmoid(self.pg),
        }

    def expected_count(self, txy, env_z, habitat_h, glm_offset):
        state_count = DENSITY_SCALE * self.state(txy)
        obs_features = self.observation_trunk(env_z)
        nonlinear_correction = 0.65 * torch.tanh(self.correction_head(obs_features))
        gate_input = torch.cat([obs_features, txy, habitat_h], dim=1)
        state_weight = 0.15 + 0.60 * torch.sigmoid(self.physics_gate(gate_input))  # floored: physics always contributes >=0.15
        log_base = torch.log(glm_offset.clamp_min(1e-4))
        log_state = torch.log(state_count.clamp_min(1e-4))
        log_expected = (
            (1.0 - state_weight) * log_base
            + state_weight * log_state
            + nonlinear_correction
        )
        expected = torch.exp(log_expected.clamp(-7.0, math.log(40.0)))
        zero_prob = 0.80 * torch.sigmoid(self.zero_head(obs_features))
        return expected, zero_prob, state_count, state_weight


def zinb_nll(y, expected, zero_prob, theta):
    """ZINB likelihood parameterized so `expected` remains E[Y]."""
    eps = 1e-7
    pi = zero_prob.clamp(eps, 0.80)
    mu = (expected / (1.0 - pi)).clamp(1e-5, 100.0)
    theta = theta.clamp(1e-3, 100.0)
    log_nb = (
        torch.lgamma(y + theta) - torch.lgamma(theta) - torch.lgamma(y + 1.0)
        + theta * (torch.log(theta) - torch.log(theta + mu))
        + y * (torch.log(mu) - torch.log(theta + mu))
    )
    log_zero = torch.logaddexp(torch.log(pi), torch.log1p(-pi) + log_nb)
    log_nonzero = torch.log1p(-pi) + log_nb
    return -torch.where(y < 0.5, log_zero, log_nonzero).mean()


def gradients(value, coordinates):
    return torch.autograd.grad(value, coordinates, torch.ones_like(value), create_graph=True)[0]


def pde_loss(model, batch):
    txy = batch["txy"].detach().clone().requires_grad_(True)
    n = model.state(txy)
    first = gradients(n, txy)
    nt, nx, ny = first[:, 0:1], first[:, 1:2], first[:, 2:3]
    nxx = gradients(nx, txy)[:, 1:2]
    nyy = gradients(ny, txy)[:, 2:3]
    h, hx, hy = batch["h"], batch["hx"], batch["hy"]
    p = model.physics_parameters()
    diffusion = p["D0"] * (1.0 - p["alpha"] * h)
    derivative = -p["D0"] * p["alpha"]
    divergence = diffusion * (nxx + nyy) + derivative * (hx * nx + hy * ny)
    growth = p["rmax"] * (p["beta"] + (1.0 - p["beta"]) * h)
    capacity = p["Kmax"] * (p["gamma"] + (1.0 - p["gamma"]) * h)
    residual = nt - divergence - growth * n * (1.0 - n / capacity)
    return (residual ** 2).mean()


def initial_condition_loss(model, batch):
    """Mean squared IC error in the dimensionless population state."""
    return ((model.state(batch["txy"]) - batch["target"]) ** 2).mean()


def boundary_loss(model, txy_np, normal_np):
    txy = torch.as_tensor(txy_np, device=DEVICE).detach().clone().requires_grad_(True)
    normal = torch.as_tensor(normal_np, device=DEVICE)
    grad = gradients(model.state(txy), txy)
    flux = grad[:, 1:2] * normal[:, 0:1] + grad[:, 2:3] * normal[:, 1:2]
    return (flux ** 2).mean()




## 5. Three-stage training without touching the official 30%

1. `data_pretrain`: data, ZINB, latent-state anchor and the full-strength
   initial-condition loss.
2. `physics_ramp`: keep the initial condition active while gradually
   introducing PDE and boundary residuals.
3. `joint_refinement`: long AdamW optimization with cosine learning-rate decay.

The complete training FIT and gate distribution are evaluated at a fixed
frequency. Hyperparameter calibration uses an inner split carved from the
official 70%; the official 30% is never monitored during training. Optional
L-BFGS uses fixed batches and `model.eval()`, so dropout is disabled and the
closure is deterministic.


The optimized objective is

\[\mathcal L=\mathcal L_{data}+\lambda_Z\mathcal L_{ZINB}+\lambda_A\mathcal L_{anchor}+\lambda_{IC}\mathcal L_{IC}+s(e)(\lambda_f\mathcal L_f+\lambda_{BC}\mathcal L_{BC}),\]

where $\lambda_{IC}=3$ follows the report and $s(e)$ ramps only the
PDE/boundary contribution. `RUN_PHYSICS_ABLATION=True` removes IC, PDE
and boundary terms together.


In [ ]:
def tensor_dict_from_pool(pool, indices):
    result = {}
    for key, values in pool.items():
        x = torch.as_tensor(values[indices], dtype=torch.float32, device=DEVICE)
        if key in {"h", "hx", "hy"}:
            x = x.reshape(-1, 1)
        result[key] = x
    return result


def prepare_stage(fit_indices, prediction_indices):
    fit_indices = np.asarray(fit_indices, dtype=np.int64)
    prediction_indices = np.asarray(prediction_indices, dtype=np.int64)
    fit_frame = df.iloc[fit_indices]
    pred_frame = df.iloc[prediction_indices]
    y_fit = fit_frame.EMBCIT.to_numpy(np.float32)

    glm = TrainingOnlyGLM().fit(fit_frame, y_fit)
    env_scaler = StandardScaler().fit(fit_frame[HYBRID_FEATURES])

    stage = {
        "fit_txy": normalized_txy(fit_frame),
        "fit_env": env_scaler.transform(fit_frame[HYBRID_FEATURES]).astype(np.float32),
        "fit_h": OBSERVATION_H[fit_indices],
        "fit_y": y_fit.reshape(-1, 1),
        "fit_glm": glm.predict(fit_frame).astype(np.float32).reshape(-1, 1),
        "pred_txy": normalized_txy(pred_frame),
        "pred_env": env_scaler.transform(pred_frame[HYBRID_FEATURES]).astype(np.float32),
        "pred_h": OBSERVATION_H[prediction_indices],
        "pred_glm": glm.predict(pred_frame).astype(np.float32).reshape(-1, 1),
        "pred_y": pred_frame.EMBCIT.to_numpy(np.float32),
        "glm_model": glm,
        "env_scaler": env_scaler,
    }
    return stage


@torch.no_grad()
def predict_hybrid(model, txy, env, habitat_h, glm_offset, batch_size=8192,
                   return_gate=False):
    model.eval()
    output, gates = [], []
    for start in range(0, len(txy), batch_size):
        sl = slice(start, start + batch_size)
        tt = torch.as_tensor(txy[sl], device=DEVICE)
        zz = torch.as_tensor(env[sl], device=DEVICE)
        hh = torch.as_tensor(habitat_h[sl], device=DEVICE)
        bb = torch.as_tensor(glm_offset[sl], device=DEVICE)
        pred, _, _, gate = model.expected_count(tt, zz, hh, bb)
        output.append(pred.cpu().numpy().ravel())
        gates.append(gate.cpu().numpy().ravel())
    prediction = np.concatenate(output)
    physics_gate = np.concatenate(gates)
    return (prediction, physics_gate) if return_gate else prediction


def _sample_indices(rng, n, batch_size):
    size = min(int(batch_size), int(n))
    return rng.choice(n, size=size, replace=False)


def _objective(model, tensors, data_idx, initial_idx, collocation_idx, boundary_idx,
               cfg, lambda_pde, physics_scale):
    pred, zero_prob, state_count, gate = model.expected_count(
        tensors["fit_txy"][data_idx],
        tensors["fit_env"][data_idx],
        tensors["fit_h"][data_idx],
        tensors["fit_glm"][data_idx],
    )
    y = tensors["fit_y"][data_idx]
    mse = ((pred - y) ** 2).mean() / tensors["y_variance"]
    theta = F.softplus(model.raw_theta) + 1e-3
    nll = zinb_nll(y, pred, zero_prob, theta)
    anchor = ((state_count - tensors["fit_glm"][data_idx]) ** 2).mean() / tensors["y_variance"]
    gate_prior = (gate.mean() - cfg.gate_prior_mean) ** 2

    zero = mse.new_zeros(())
    # L_IC is active at full strength before and during the PDE ramp.
    # A no-physics ablation passes initial_idx=None and therefore removes it.
    lic = (
        initial_condition_loss(
            model, tensor_dict_from_pool(INITIAL_POOL, initial_idx)
        )
        if initial_idx is not None else zero
    )
    if physics_scale > 0.0:
        lpde = pde_loss(model, tensor_dict_from_pool(PHYSICS_POOL, collocation_idx))
        lbc = boundary_loss(
            model,
            BOUNDARY_POOL["txy"][boundary_idx],
            BOUNDARY_POOL["normal"][boundary_idx],
        )
    else:
        lpde, lbc = zero, zero

    total = (
        mse
        + cfg.lambda_zinb * nll
        + cfg.lambda_anchor * anchor
        + cfg.lambda_gate_prior * gate_prior
        + cfg.lambda_ic * lic
        + physics_scale * (lambda_pde * lpde + cfg.lambda_bc * lbc)
    )
    return {
        "total": total,
        "mse": mse,
        "zinb": nll,
        "anchor": anchor,
        "gate_prior": gate_prior,
        "ic": lic,
        "pde": lpde,
        "bc": lbc,
        "gate_batch_mean": gate.mean(),
    }


def _phase_plan(cfg):
    return [
        ("data_pretrain", cfg.pretrain_epochs, cfg.lr_pretrain),
        ("physics_ramp", cfg.physics_ramp_epochs, cfg.lr_ramp),
        ("joint_refinement", cfg.refine_epochs, cfg.lr_refine),
    ]


def _full_metrics(model, stage, include_prediction=False):
    fit_pred, fit_gate = predict_hybrid(
        model, stage["fit_txy"], stage["fit_env"], stage["fit_h"],
        stage["fit_glm"], return_gate=True,
    )
    fit_y = stage["fit_y"].ravel()
    out = {
        "full_train_fit": FIT(fit_y, fit_pred),
        "full_train_normalized_mse": float(
            np.mean((fit_y - fit_pred) ** 2) / (np.var(fit_y) + 1e-6)
        ),
        "train_gate_mean": float(fit_gate.mean()),
        "train_gate_sd": float(fit_gate.std(ddof=0)),
        "train_gate_min": float(fit_gate.min()),
        "train_gate_max": float(fit_gate.max()),
    }
    if include_prediction:
        pred, gate = predict_hybrid(
            model, stage["pred_txy"], stage["pred_env"], stage["pred_h"],
            stage["pred_glm"], return_gate=True,
        )
        out.update({
            "inner_valid_fit": FIT(stage["pred_y"], pred),
            "valid_gate_mean": float(gate.mean()),
            "valid_gate_sd": float(gate.std(ddof=0)),
        })
    return out


def train_hybrid(stage, cfg, seed, lambda_pde=None, monitor=False,
                 verbose=False, with_physics=True, use_lbfgs=None):
    """Three-stage optimization.

    `monitor=True` is reserved for an inner validation split carved from the
    outer 70%. With `monitor=False`, evaluation every `eval_every` epochs uses
    the full training set only and never touches the official outer 30%. The
    report's paper-IC target is active from the first pretraining epoch.
    """
    seed_everything(seed)
    model = HybridPINN(len(HYBRID_FEATURES), cfg, seed).to(DEVICE)
    lambda_pde = cfg.lambda_pde if lambda_pde is None else float(lambda_pde)
    use_lbfgs = cfg.use_lbfgs if use_lbfgs is None else bool(use_lbfgs)
    if not with_physics:
        lambda_pde, use_lbfgs = 0.0, False

    tensors = {
        "fit_txy": torch.as_tensor(stage["fit_txy"], device=DEVICE),
        "fit_env": torch.as_tensor(stage["fit_env"], device=DEVICE),
        "fit_h": torch.as_tensor(stage["fit_h"], device=DEVICE),
        "fit_y": torch.as_tensor(stage["fit_y"], device=DEVICE),
        "fit_glm": torch.as_tensor(stage["fit_glm"], device=DEVICE),
        "y_variance": float(np.var(stage["fit_y"])) + 1e-6,
    }
    rng = np.random.default_rng(seed)

    best_fit, best_epoch, best_state = -np.inf, cfg.adam_epochs, None
    history = []
    global_epoch = 0

    for phase_name, phase_epochs, learning_rate in _phase_plan(cfg):
        if phase_epochs <= 0:
            continue
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=learning_rate, weight_decay=cfg.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(phase_epochs, 1), eta_min=learning_rate * 0.03
        )

        for local_epoch in range(1, phase_epochs + 1):
            global_epoch += 1
            model.train()
            di = _sample_indices(rng, len(tensors["fit_y"]), cfg.data_batch)
            ii = (
                _sample_indices(rng, len(INITIAL_POOL["txy"]), cfg.initial_batch)
                if with_physics else None
            )

            if not with_physics or phase_name == "data_pretrain":
                physics_scale = 0.0
                ci = bi = None
            else:
                ci = _sample_indices(rng, len(PHYSICS_POOL["txy"]), cfg.physics_batch)
                bi = _sample_indices(rng, len(BOUNDARY_POOL["txy"]), cfg.boundary_batch)
                physics_scale = (
                    local_epoch / max(1, phase_epochs)
                    if phase_name == "physics_ramp" else 1.0
                )

            terms = _objective(
                model, tensors, di, ii, ci, bi, cfg, lambda_pde, physics_scale
            )
            optimizer.zero_grad(set_to_none=True)
            terms["total"].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            scheduler.step()

            row = {
                "epoch": global_epoch,
                "phase": phase_name,
                "physics_scale": physics_scale,
                **{k: float(v.detach()) for k, v in terms.items()},
            }

            evaluate_now = (
                global_epoch % cfg.eval_every == 0 or local_epoch == phase_epochs
            )
            if evaluate_now:
                metrics = _full_metrics(model, stage, include_prediction=monitor)
                row.update(metrics)
                monitor_fit = metrics.get("inner_valid_fit", np.nan)
                if verbose:
                    valid_text = (
                        f" inner FIT={monitor_fit:.3f}" if monitor else ""
                    )
                    print(
                        f"  {phase_name:16s} epoch {global_epoch:4d}/{cfg.adam_epochs}: "
                        f"loss={row['total']:.4f} data={row['mse']:.4f} "
                        f"ic={row['ic']:.4f} pde={row['pde']:.4f} "
                        f"train FIT={metrics['full_train_fit']:.3f} "
                        f"gate={metrics['train_gate_mean']:.3f}+/-{metrics['train_gate_sd']:.3f}"
                        f"{valid_text}"
                    )
                if monitor and monitor_fit > best_fit + 1e-4:
                    best_fit, best_epoch = monitor_fit, global_epoch
                    best_state = copy.deepcopy(
                        {k: v.detach().cpu() for k, v in model.state_dict().items()}
                    )
            history.append(row)

    # Optional deterministic polish. Dropout is disabled by model.eval(), and
    # the same fixed data/collocation/boundary subsets are reused by every
    # closure evaluation, which is required by L-BFGS line search.
    if use_lbfgs and with_physics and cfg.lbfgs_steps > 0:
        model.eval()
        di = _sample_indices(rng, len(tensors["fit_y"]), cfg.lbfgs_data)
        ci = _sample_indices(rng, len(PHYSICS_POOL["txy"]), cfg.lbfgs_physics)
        bi = _sample_indices(rng, len(BOUNDARY_POOL["txy"]), cfg.lbfgs_boundary)
        ii = _sample_indices(rng, len(INITIAL_POOL["txy"]), cfg.lbfgs_initial)
        optimizer = torch.optim.LBFGS(
            model.parameters(), lr=0.5, max_iter=1, max_eval=6,
            history_size=50, line_search_fn="strong_wolfe",
        )
        last_terms = {}

        def closure():
            nonlocal last_terms
            optimizer.zero_grad(set_to_none=True)
            last_terms = _objective(
                model, tensors, di, ii, ci, bi, cfg, lambda_pde, physics_scale=1.0
            )
            last_terms["total"].backward()
            return last_terms["total"]

        for lb_step in range(1, cfg.lbfgs_steps + 1):
            optimizer.step(closure)
            if lb_step % 5 == 0 or lb_step == cfg.lbfgs_steps:
                metrics = _full_metrics(model, stage, include_prediction=monitor)
                global_epoch += 1
                row = {
                    "epoch": global_epoch,
                    "phase": "lbfgs",
                    "lbfgs_step": lb_step,
                    "physics_scale": 1.0,
                    **{k: float(v.detach()) for k, v in last_terms.items()},
                    **metrics,
                }
                history.append(row)
                monitor_fit = metrics.get("inner_valid_fit", np.nan)
                if verbose:
                    print(
                        f"  lbfgs step {lb_step:3d}/{cfg.lbfgs_steps}: "
                        f"loss={row['total']:.4f} ic={row['ic']:.4f} "
                        f"pde={row['pde']:.4f} train FIT={metrics['full_train_fit']:.3f}" +
                        (f" inner FIT={monitor_fit:.3f}" if monitor else "")
                    )
                if monitor and monitor_fit > best_fit + 1e-4:
                    best_fit, best_epoch = monitor_fit, global_epoch
                    best_state = copy.deepcopy(
                        {k: v.detach().cpu() for k, v in model.state_dict().items()}
                    )

    if monitor and best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch, best_fit


def _short_tuning_config(cfg):
    tuned = copy.deepcopy(cfg)
    tuned.pretrain_epochs = cfg.tune_pretrain_epochs
    tuned.physics_ramp_epochs = cfg.tune_ramp_epochs
    tuned.refine_epochs = cfg.tune_refine_epochs
    tuned.use_lbfgs = cfg.use_lbfgs
    tuned.lbfgs_steps = min(cfg.lbfgs_steps, 5)
    tuned.run_physics_ablation = False
    return tuned


def calibrate_lambda_pde(cfg):
    """Select lambda_PDE using only inner splits from the official 70%."""
    candidates = tuple(float(x) for x in cfg.pde_candidates)
    if len(candidates) == 1:
        print(f"fixed inner-training lambda_PDE={candidates[0]:.4g}")
        return candidates[0], pd.DataFrame()

    calibration_draws = [0] if cfg.mode == "pilot" else [0, 1, 2]
    tune_cfg = _short_tuning_config(cfg)
    records = []
    for draw in calibration_draws:
        outer_train = EST_SPLITS[draw]
        years = df.iloc[outer_train].Year.to_numpy()
        inner_train, inner_valid = train_test_split(
            outer_train,
            test_size=cfg.inner_fraction,
            random_state=cfg.seed + draw,
            stratify=years,
        )
        stage = prepare_stage(inner_train, inner_valid)
        for value in candidates:
            _, _, best_epoch, best_fit = train_hybrid(
                stage,
                tune_cfg,
                seed=cfg.seed + draw,
                lambda_pde=value,
                monitor=True,
                verbose=(draw == calibration_draws[0]),
                with_physics=True,
                use_lbfgs=tune_cfg.use_lbfgs,
            )
            records.append({
                "draw": draw,
                "lambda_pde": value,
                "best_epoch": best_epoch,
                "inner_valid_fit": best_fit,
            })
            print(
                f"calibration draw={draw} lambda_PDE={value:.4g}: "
                f"best inner FIT={best_fit:.3f} at epoch {best_epoch}"
            )

    table = pd.DataFrame(records)
    score = table.groupby("lambda_pde")["inner_valid_fit"].mean()
    selected = float(score.idxmax())
    print("mean inner-validation FIT by lambda_PDE:")
    print(score.to_string())
    print(f"selected and frozen lambda_PDE={selected:.4g}")
    return selected, table




## 6. Official PINN versus Random Forest test

Run `smoke`, then `pilot`. Inspect only the **inner-training** calibration
output, freeze the configuration, restart the runtime, and run `final` once.
Do not use the official 30% results to choose another configuration.



In [ ]:
def run_one_draw(draw, cfg, lambda_pde):
    train_idx, valid_idx = EST_SPLITS[draw], VAL_SPLITS[draw]
    stage = prepare_stage(train_idx, valid_idx)
    y_train = df.iloc[train_idx].EMBCIT.to_numpy(float)
    y_valid = df.iloc[valid_idx].EMBCIT.to_numpy(float)

    # The seed mapping is intentionally identical to inner calibration. The old
    # notebook added 10,000 here, which changed the Fourier map at refit time.
    model, history, _, _ = train_hybrid(
        stage,
        cfg,
        seed=cfg.seed + draw,
        lambda_pde=lambda_pde,
        monitor=False,
        verbose=False,
        with_physics=True,
        use_lbfgs=cfg.use_lbfgs,
    )
    pinn_train, gate_train = predict_hybrid(
        model, stage["fit_txy"], stage["fit_env"], stage["fit_h"],
        stage["fit_glm"], return_gate=True,
    )
    pinn_valid, gate_valid = predict_hybrid(
        model, stage["pred_txy"], stage["pred_env"], stage["pred_h"],
        stage["pred_glm"], return_gate=True,
    )

    control = fit_random_forest(
        df.iloc[train_idx], y_train, cfg.seed + draw,
        smoke=cfg.mode == "smoke", use_year=cfg.rf_use_year,
    )
    rf_features = control.feature_names_used_
    control_train = control.predict(df.iloc[train_idx][rf_features])
    control_valid = control.predict(df.iloc[valid_idx][rf_features])

    glm_train = stage["fit_glm"].ravel()
    glm_valid = stage["pred_glm"].ravel()
    p = {k: float(v.detach()) for k, v in model.physics_parameters().items()}
    with torch.no_grad():
        ic_batch = tensor_dict_from_pool(
            INITIAL_POOL, np.arange(len(INITIAL_POOL["txy"]))
        )
        final_ic_loss = float(initial_condition_loss(model, ic_batch).cpu())
    row = {
        "draw": draw,
        "n_train": len(train_idx),
        "n_valid": len(valid_idx),
        "adam_epochs": cfg.adam_epochs,
        "lbfgs_enabled": cfg.use_lbfgs,
        "lambda_pde": lambda_pde,
        "lambda_ic": cfg.lambda_ic,
        "initial_condition_loss": final_ic_loss,
        "initial_condition_rmse_counts": DENSITY_SCALE * math.sqrt(final_ic_loss),
        "rf_uses_year": cfg.rf_use_year,
        "glm_train_fit": FIT(y_train, glm_train),
        "glm_valid_fit": FIT(y_valid, glm_valid),
        "rf_train_fit": FIT(y_train, control_train),
        "rf_valid_fit": FIT(y_valid, control_valid),
        "pinn_train_fit": FIT(y_train, pinn_train),
        "pinn_valid_fit": FIT(y_valid, pinn_valid),
        "pinn_minus_rf_valid_fit": FIT(y_valid, pinn_valid) - FIT(y_valid, control_valid),
        "gate_train_mean": float(gate_train.mean()),
        "gate_train_sd": float(gate_train.std(ddof=0)),
        "gate_valid_mean": float(gate_valid.mean()),
        "gate_valid_sd": float(gate_valid.std(ddof=0)),
        "gate_valid_min": float(gate_valid.min()),
        "gate_valid_max": float(gate_valid.max()),
        "D0": p["D0"], "alpha": p["alpha"], "rmax": p["rmax"],
        "beta": p["beta"], "Kmax": p["Kmax"], "gamma": p["gamma"],
    }

    if cfg.run_physics_ablation:
        ablation, _, _, _ = train_hybrid(
            stage,
            cfg,
            seed=cfg.seed + draw,
            lambda_pde=0.0,
            monitor=False,
            verbose=False,
            with_physics=False,
            use_lbfgs=False,
        )
        no_physics_train = predict_hybrid(
            ablation, stage["fit_txy"], stage["fit_env"], stage["fit_h"], stage["fit_glm"]
        )
        no_physics_valid = predict_hybrid(
            ablation, stage["pred_txy"], stage["pred_env"], stage["pred_h"], stage["pred_glm"]
        )
        row.update({
            "no_physics_train_fit": FIT(y_train, no_physics_train),
            "no_physics_valid_fit": FIT(y_valid, no_physics_valid),
            "physics_gain_valid_fit": (
                FIT(y_valid, pinn_valid) - FIT(y_valid, no_physics_valid)
            ),
        })
    return row, history


def run_benchmark(cfg):
    cfg.configure_mode()
    draws = [0] if cfg.mode == "smoke" else list(range(10 if cfg.mode == "pilot" else 100))
    selected_lambda, calibration = calibrate_lambda_pde(cfg)

    print(
        f"frozen settings: AdamW epochs={cfg.adam_epochs}, "
        f"lambda_PDE={selected_lambda:.4g}, lambda_IC={cfg.lambda_ic:g}, "
        f"L-BFGS={cfg.use_lbfgs}, RF Year={cfg.rf_use_year}"
    )

    output_dir = Path("yellowhammer_pinn_vs_rf_outputs")
    output_dir.mkdir(exist_ok=True)
    output_csv = output_dir / f"pinn_vs_rf_official_70_30_{cfg.mode}.csv"
    if len(calibration):
        calibration.to_csv(output_dir / f"inner_calibration_{cfg.mode}.csv", index=False)
    rows = []
    for draw in draws:
        start = time.time()
        row, history = run_one_draw(draw, cfg, selected_lambda)
        row["elapsed_seconds"] = time.time() - start
        rows.append(row)
        pd.DataFrame(rows).to_csv(output_csv, index=False)
        if draw == draws[0]:
            history.to_csv(output_dir / f"training_history_draw_{draw:02d}_{cfg.mode}.csv", index=False)
        print(
            f"draw {draw:02d}: RF={row['rf_valid_fit']:.3f} | "
            f"adaptive PINN={row['pinn_valid_fit']:.3f} | "
            f"delta={row['pinn_minus_rf_valid_fit']:+.3f} | "
            f"gate={row['gate_valid_mean']:.3f}+/-{row['gate_valid_sd']:.3f}"
        )
    return pd.DataFrame(rows), output_csv


results, result_file = run_benchmark(CFG)
print("saved", result_file)




In [ ]:
def summary_table(results):
    rf_label = (
        "Random Forest (same inputs)" if CFG.rf_use_year
        else "Random Forest (report inputs)"
    )
    mapping = {
        rf_label: "rf_valid_fit",
        "Adaptive-gated Hybrid PINN": "pinn_valid_fit",
    }
    if "no_physics_valid_fit" in results:
        mapping["Same network without PDE (ablation)"] = "no_physics_valid_fit"
    rows = []
    for name, column in mapping.items():
        rows.append({
            "model": name,
            "draws": len(results),
            "validation_FIT_mean": results[column].mean(),
            "validation_FIT_sample_sd": results[column].std(ddof=1),
        })
    return pd.DataFrame(rows)


display(summary_table(results))

paired = pd.DataFrame({
    "comparison": ["Adaptive PINN - Random Forest"],
    "mean_paired_FIT_difference": [results.pinn_minus_rf_valid_fit.mean()],
    "PINN_wins": [(results.pinn_minus_rf_valid_fit > 0).sum()],
    "draws": [len(results)],
})
display(paired)

if len(results) > 1:
    labels = ["Random Forest", "Adaptive-gated PINN"]
    columns = ["rf_valid_fit", "pinn_valid_fit"]
    means = [results[c].mean() for c in columns]
    errors = [results[c].std(ddof=1) for c in columns]
    plt.figure(figsize=(6.4, 4.5))
    plt.bar(labels, means, yerr=errors, capsize=4)
    plt.axhline(19.3, color="tab:red", linestyle="--", label="Report RF = 19.3")
    plt.ylabel("Official validation FIT (%)")
    plt.title(f"Yellowhammer: PINN versus RF ({len(results)} official draws)")
    plt.legend()
    plt.tight_layout()
    plt.show()




## How to interpret the result

- The primary evidence is the paired quantity `pinn_minus_rf_valid_fit` over
  all 100 official draws, not the training loss.
- With `RF_USE_YEAR=True`, RF and the PINN observation head receive the same
  tabular variables. With `False`, RF uses the report's original inputs.
- Set `RUN_PHYSICS_ABLATION=True` for the publication ablation. A positive
  `physics_gain_valid_fit` is required before claiming that the PDE itself
  improves prediction.
- The adaptive gate is diagnosed by its mean, standard deviation and range.
  A gate that is almost always zero means that the predictive model is
  ignoring the latent physics even if the PDE residual is small.
- `initial_condition_loss` is the dimensionless MSE at the 2,500 fixed
  $t=2002$ points; `initial_condition_rmse_counts` converts it back to the
  original count scale. Both should decrease without forcing data FIT down.
- `paper-H` and `paper-IC` must be disclosed because both are derived from
  `u_GLM`. A strict leakage-free publication result requires exogenous or
  per-split constructions.


In [ ]:
import shutil
from google.colab import files

# Compresser tout le contenu de /content
shutil.make_archive('/tmp/all_files', 'zip', '/content')

# Télécharger le ZIP
files.download('/tmp/all_files.zip')